# Planners

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/08_planners.ipynb)

Every planner in minilink reads a `PlanningProblem` and returns a `PlanningSolution` from `solve()`. This tutorial presents the public API of `Planner` (`TrajectoryOptimizationPlanner`, `LQRPlanner`, `DynamicProgrammingPlanner`), `PlanningSolution`, `Comparison`, the pure compiled planner face `planner.compile()`, and `ModelPredictiveController`.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `Planner`: one verb, `solve`

A planner is built on a `PlanningProblem` with the method's own choices, and `solve()` returns a `PlanningSolution`. The problem stays declarative: it does not solve itself.

### The base class

| Call | Returns | Meaning |
| --- | --- | --- |
| `planner.problem` | `PlanningProblem` | The problem the planner was built on |
| `planner.params` | `dict` | The planner's tree: the problem's leaves by role (live), the method's numbers under `"method"`, and `"seed"` (`None` for a planner that draws nothing) |
| `planner.dt` | `float \| None` | The planner's own step, on which its views run a feedback law (`None`: the evaluator's automatic step) |
| `planner.solve(...)` | `PlanningSolution` | Solve, and keep the solution as `planner.last_solution`. Each planner has its own keywords (below); all take `params=None`, the complete tree of `planner.params` edited (`copy.deepcopy(planner.params)`), and `evaluate=False`, `n_trials=50`, which also score the controller over the problem's draws |
| `planner.last_solution` | `PlanningSolution \| None` | The object the latest `solve()` returned |
| `planner.controller` | `System` | The latest solution's controller |
| `planner.cost_to_go(x, t=None)` | `float` | The latest solution's cost-to-go read at `x` (at `t` when given) |
| `planner.plot_trajectory(**kwargs)`, `planner.animate(**kwargs)`, `planner.plot_cost(**kwargs)` | as on the solution | The latest solution's views, a feedback law run on `planner.dt` |
| `planner.plot_control_law(...)`, `planner.plot_cost_to_go(...)` | `PlotResult` | The latest solution's law and cost-to-go, the same keywords as on the solution |

### The planners

| Planner | Constructor | `solve` | Controller | `cost_to_go` |
| --- | --- | --- | --- | --- |
| `TrajectoryOptimizationPlanner` | `TrajectoryOptimizationPlanner(problem, *, transcription=None, n_steps=None, method="slsqp", backend="numpy", n_iterations=None, use_hessian=False, feasibility_tol=1e-6, guess=None, options=None, **method_numbers)`; `transcription` is `"direct_collocation"` (the default), `"multiple_shooting"` or `"single_shooting"` | `trajopt.solve(*, params=None, guess=None, evaluate=False, n_trials=50, verbose=False, callback=None, live_plot=False, record_history=False)` | open loop, a `TrajectorySource` | `None` |
| `LQRPlanner` | `LQRPlanner(problem, *, x_bar=None, u_bar=None, dt=0.01, n_samples=1001, t=0.0, method="auto", eps=1e-6)` | `lqr_planner.solve(*, params=None, evaluate=False, n_trials=50, verbose=False)` | feedback, $u = \bar u - K (x - \bar x)$ | a `QuadraticField` |
| `DynamicProgrammingPlanner` | `DynamicProgrammingPlanner(problem, *, grid=None, x_grid=None, u_grid=None, dt=None, solver="euler", method="table", backend="numpy", gamma=None, tol=0.1, max_iterations=1000, interpolation="linear", out_of_bound_cost=None, clean_infeasible=True)` | `vi.solve(*, params=None, evaluate=False, n_trials=50, verbose=False, record_history=False)` | feedback, a table lookup | a `GridField` |
| `MPPIPlanner` (`minilink.planning`) | `MPPIPlanner(problem, *, dt=0.05, n_rollouts=1024, temperature=1.0, noise=None, control_cost=0.0, n_iterations=1, solver="rk4", seed=0)` | `solve(*, params=None, guess=None, evaluate=False, n_trials=50, verbose=False)` | open loop | `None` |
| `RRTPlanner`, `RRTStarPlanner` | `RRTPlanner(problem, extender=None, *, ...)` | `solve(*, params=None, evaluate=False, n_trials=50, verbose=False, callback=None)` | open loop | `None` |
| `TabularLearningPlanner` | `TabularLearningPlanner(problem, *, ...)` | `solve(n_episodes=1000, *, params=None, evaluate=False, n_trials=50, verbose=False)` | feedback, a table lookup | a `GridField` |
| `ReinforcementLearningPlanner` | `ReinforcementLearningPlanner(problem, *, ...)` | `solve(n_timesteps=100000, *, params=None, evaluate=False, n_trials=50, verbose=False)` | feedback, a neural policy | the critic's field |

`TrajectoryOptimizationPlanner` and `MPPIPlanner` also have `planner.compile(backend=None)`, the planner's pure face (section 4). `DynamicProgrammingPlanner` adds `solve_steps(n_steps, ...)`, `evaluate_policy(controller)` and `tables`, the planner's own grid tables.

## 2. `PlanningSolution`: what every planner returns

A frozen record: the controller, the plan it predicts, and the evidence for them. One rule: the controller handed out is the block that `evaluation` scores and the views draw, as it is.

| Field / call | Type | Meaning |
| --- | --- | --- |
| `sol.problem` | `PlanningProblem` | What was solved |
| `sol.controller` | `System` | The block to run. A feedback law reads the state: `controller @ plant` closes the loop. An open-loop plan is a source with no input, such as a `TrajectorySource`: `controller >> plant` drives the plant |
| `sol.record` | the planner's record | The planner's account of the solve, one dataclass per planner (`TrajectoryOptimizationRecord`, `RiccatiRecord`, `ValueIterationRecord`, `TreeSearchRecord`, `MPPIRecord`, `TabularLearningRecord`, `ReinforcementLearningRecord`), each with a `success` flag and a one-line `str` |
| `sol.trajectory` | `Trajectory \| None` | The plan an open-loop planner computed, on its own grid; `None` for a feedback law |
| `sol.cost_to_go` | `Field \| None` | $J(x)$, the method's own estimate where it produces one, called `J(x)` or `J(x, t=t)` |
| `sol.evaluation` | `Evaluation \| None` | The score of `controller` over the problem's draws, when the solve was asked to evaluate |
| `sol.params` | `dict` | The numbers the solve ran on, a copy of the planner's tree: `planner.solve(params=sol.params)` solves it again |
| `sol.success` | `bool` | The record's own success |
| `sol.open_loop` | `bool` | Whether the controller is a source with no input, $u = \pi(t)$ |
| `sol.family` | `str` | The planning family, read from the record's class name (`"trajectory optimization"`, `"riccati"`, `"value iteration"`, ...) |
| `sol.plot_trajectory(*, dt=None, signals=("x", "u"), renderer="matplotlib", show=True)` | `PlotResult` | Draw the plan, else the controller's run from the problem's start (integrated on `dt`), titled plan or run |
| `sol.animate(*, dt=None, **kwargs)` | as `sys.animate` | Animate the same plan or run on the problem's system |
| `sol.plot_cost(*, dt=None, renderer="matplotlib", show=True)` | `PlotResult` | The running cost and $J(t)$ along the plan or run |
| `sol.plot_control_law(*, x_axis=None, y_axis=None, u_axis=0, anchor=None, bounds=None, grid_shape=(101, 101), t=0.0, is_3d=False, trajectory=None, title=None, ax=None, renderer="matplotlib", show=True)` | `PlotResult` | A slice of a feedback law over two state axes |
| `sol.plot_cost_to_go(*, x_axis=0, y_axis=1, anchor=None, bounds=None, grid_shape=(51, 51), t=0.0, jmax=None, is_3d=False, trajectory=None, title=None, ax=None, renderer="matplotlib", show=True)` | `PlotResult` | A slice of `cost_to_go` over two state axes |

In [ ]:
import numpy as np

from minilink import Pendulum, PlanningProblem, QuadraticCost, TrajectoryOptimizationPlanner

plant = Pendulum()
UP = np.array([np.pi, 0.0])
cost = QuadraticCost.from_system(plant, x_bar=UP, Q=np.eye(2), R=0.1 * np.eye(1))
problem = PlanningProblem(plant, x0=np.zeros(2), cost=cost, U=plant.input_box, Xf=UP, horizon=5.0)

trajopt = TrajectoryOptimizationPlanner(
    problem, n_steps=20, transcription="direct_collocation", backend="jax"
)
plan = trajopt.solve()
print(plan)

In [ ]:
(
    plan.success,
    plan.open_loop,
    plan.family,
    type(plan.controller).__name__,
    plan.trajectory.x.shape,
    plan.record.J,
    list(plan.params),
)

In [ ]:
plan.plot_trajectory();

## 3. Feedback planners, their cost-to-go, and `compare`

`LQRPlanner` and `DynamicProgrammingPlanner` return a feedback law $u = \pi(x)$ and a cost-to-go `sol.cost_to_go`, a `Field`. Here both solve the regulation of the pendulum about the upright on an infinite horizon; value iteration plans on the problem discounted at $\rho = 0.2$, so that its sweeps converge.

In [ ]:
from minilink import DynamicProgrammingPlanner, LQRPlanner

regulation = problem.replace(x0=np.array([np.pi - 0.2, 0.0]), horizon=None, Xf=None)

lqr_solution = LQRPlanner(regulation).solve()
value_iteration = DynamicProgrammingPlanner(
    regulation.replace(discount_rate=0.2), x_grid=(81, 81), u_grid=(11,), dt=0.05, tol=0.1
)
vi_solution = value_iteration.solve()
print(vi_solution)

In [ ]:
# The cost-to-go at the bottom and at the upright
(
    vi_solution.cost_to_go(np.zeros(2)),
    vi_solution.cost_to_go(UP),
    lqr_solution.cost_to_go(np.zeros(2)),
)

`compare` ([`06_compile_and_evaluate`](06_compile_and_evaluate.ipynb)) takes solutions as entries. With the problem first it scores every controller on that problem, here the undiscounted one; `episode_length` is the length of the run of an infinite-horizon problem. Its `Comparison` overlays the laws and the cost-to-go fields on one colour scale.

In [ ]:
from minilink import compare

comparison = compare(regulation, LQR=lqr_solution, VI=vi_solution, dt=0.05, episode_length=8.0)
print(comparison)

In [ ]:
comparison.plot_control_law();

In [ ]:
comparison.plot_cost_to_go();

A solution's controller is a `System`: `@` closes it on the plant, here from the bottom.

In [ ]:
vi_loop = vi_solution.controller @ plant
vi_traj = vi_loop.compute_trajectory(tf=8.0, x0=np.zeros(vi_loop.n))

In [ ]:
vi_loop.plot_trajectory(vi_traj);

## 4. `CompiledTrajectoryPlanner`: the planner's pure face

`pv = planner.compile(backend=None)` lowers a trajectory-optimization planner to functions of `(params, z)`. `z` is the planner's solution vector: the transcription's decision vector (the knots) with the origin time `t0` it was solved from appended, `pv.n_z` entries. The face keeps no last solution: every answer is a function of its arguments, and with a native method and `n_iterations=K` on JAX it traces under `jit`, `vmap` and `grad`.

| Call | Returns | Meaning |
| --- | --- | --- |
| `pv.n_z` | `int` | Size of `z` |
| `pv.knot_spacing` | `float` | The time between the plan's knots |
| `pv.predicts` | `bool` | Whether `z` carries the state knots (direct collocation) |
| `pv.params` | `dict` | The compiled defaults, a copy: `"x0"`, `"t0"`, the plant's, the cost's, the discount, the sets read as constraints, `"method"` and `"seed"` |
| `pv.initial(params=None)` | `array (n_z,)` | The default guess at `params` |
| `pv.solve(params=None, z0=None)` | `(z, status)` | Solve from the guess `z0` (default `pv.initial(params)`) at `params`; `status` is `{"success", "n_iterations", "J"}` |
| `pv.command(z, x, t, params=None)` | `array (m,)` | The input the plan gives at time `t`, linear between knots (an open-loop plan ignores `x`) |
| `pv.shift(z, t, x0=None)` | `array (n_z,)` | The plan rebased to start at the time `t`, its tail held; with `x0`, a plan that carries its states has its first state knot pinned there |
| `pv.predict_start(x, u, t, delta, params=None)` | `array (n,)` | $\hat x(t + \Delta)$ on the planner's model, `u` held: the start of a solve that compensates its own delay |
| `pv.prediction(z, t, params=None)` | `(x, u)` | The planned state and input at `t` (a plan that carries its states) |
| `pv.planned_trajectory(z, params=None)` | `Trajectory` | The plan's knots as a trajectory (a plan that carries its states) |
| `pv.solution(z, params=None, status=None)` | `PlanningSolution` | The full solution of `z`, rebuilt on the host |
| `pv.pack(guess, params=None)` | `array (n_z,)` | A `Trajectory` or a `PlanningSolution` in `z` form |

In [ ]:
x_tilt = np.array([np.pi - 0.5, 0.0])
task = PlanningProblem(plant, x0=x_tilt, cost=cost, U=plant.input_box, horizon=4.0)

short_planner = TrajectoryOptimizationPlanner(
    task.replace(horizon=1.0),
    n_steps=10,
    transcription="single_shooting",
    backend="jax",
    method="adam",
    n_iterations=30,
    learning_rate=0.2,
)
pv = short_planner.compile()
pv.n_z, pv.knot_spacing, pv.predicts, list(pv.params)

In [ ]:
z0 = pv.initial()
z, status = pv.solve(z0=z0)
u_now = pv.command(z, x_tilt, t=0.0)

# One receding-horizon tick by hand: predict the state 0.1 s ahead, rebase the plan
# there, and solve again from it
x_pred = pv.predict_start(x_tilt, u_now, t=0.0, delta=0.1)
z_warm = pv.shift(z, t=0.1)
tick = pv.params
tick["x0"] = x_pred
tick["t0"] = 0.1
z_next, status_next = pv.solve(params=tick, z0=z_warm)
status, u_now, x_pred, status_next

In [ ]:
# The full PlanningSolution of z, rebuilt on the host
rebuilt = pv.solution(z, status=status)
rebuilt.trajectory.x.shape, rebuilt.record.J

## 5. `ModelPredictiveController`: the planner in the loop

`ModelPredictiveController(planner, *, inputs=None, delay=False, warm_start=True, backend=None, verbose=False)` (`minilink.control`) is a step diagram of three pure blocks around `planner.compile()`: `solve` (the plan from the measured state and the last plan), `command` (the plan read at the tick's time), and `memory` (a unit delay holding the last plan).

| Argument | Meaning |
| --- | --- |
| `planner` | A `TrajectoryOptimizationPlanner` or an `MPPIPlanner`, on a problem whose start is a point |
| `inputs` | Leaves of the planner's tree read from input ports, `{path: port}`; the default `{"x0": "y"}` binds the plan's start to the measured state |
| `warm_start` | `True`: each solve starts from the last plan shifted to the tick's time; `False`: from the default guess |
| `delay` | `True`: tick $k$ applies the plan solved at tick $k - 1$, one period of computation delay; `False`: the plan is applied at once |
| `backend` | The compiled face's backend (default the planner's) |
| `verbose` | Print one line per solve |

Its output ports are `"u"` (the command) and `"z"` (the plan), and `"x_ref"`, `"u_ref"` when the plan carries its states; `mpc.pv` is the compiled face. `mpc % dt` puts it on a clock, and `(mpc % dt) @ plant`, or `problem.compile(mpc % dt)`, closes it on the plant:

In [ ]:
from minilink.control import ModelPredictiveController

mpc = ModelPredictiveController(short_planner)
list(mpc.subsystems), list(mpc.inputs), list(mpc.outputs)

In [ ]:
pe = task.compile(mpc % 0.1, substeps=1, backend="jax")
print(pe.evaluate())

In [ ]:
mpc_traj = pe.trajectory()

In [ ]:
pe.loop.plot_trajectory(mpc_traj);

## Summary: the core object pipeline

1. **`System` & `Trajectory`** ([`01_systems`](01_systems.ipynb)): continuous (`DynamicSystem`), composite (`DiagramSystem`), discrete (`StepSystem`), and clocked (`SampledSystem`, `SampledDiagramSystem`) blocks, their `Trajectory` records, the `Simulator`, and the graphics contract.
2. **`.params`** ([`02_parameters`](02_parameters.ipynb)): uniform dictionary/tree parameter protocol across every object, functional overrides, and `realize(key, n)`.
3. **`Set`, `InputSet`, `Field`, `Distribution`** ([`03_sets_fields_distributions`](03_sets_fields_distributions.ipynb)): feasibility margins (`margin`, `contains`), scalar fields (`value`, `gradient`, `as_set`, `as_cost`), and probability laws (`sample`, `mean`, `cov`, `support`).
4. **`CostFunction` & `StepCost`** ([`04_costs`](04_costs.ipynb)): running/terminal costs (`g`, `h`, `integrate`, `g_max`, `h_max`), cost algebra (`SumCost`, `ScaledCost`), and sampled stage costs (`SampledCost`).
5. **`PlanningProblem` & `StepPlanningProblem`** ([`05_problem`](05_problem.ipynb)): task specification, `replace`, stochastic realizations, `discretize`, and the step problem's own equations (`transition`, `terminal`, `rollout`).
6. **`ProblemEvaluator`, `Evaluation`, the compiled evaluators** ([`06_compile_and_evaluate`](06_compile_and_evaluate.ipynb)): `problem.compile(controller)`, differentiable `Evaluation` scores, Monte Carlo batches, `compare`, and `sys.compile()`.
7. **`MathematicalProgram` & `Optimizer`** ([`07_program`](07_program.ipynb)): `pe.program(wrt)`, `pack`/`unpack`, general NLPs, parametric solves, and differentiable fixed-step optimizers.
8. **`Planner`, `PlanningSolution`, `CompiledTrajectoryPlanner`, `ModelPredictiveController`** ([`08_planners`](08_planners.ipynb)): offline planning, value functions, side-by-side comparisons, pure compiled planner primitives, and receding-horizon MPC.